In [93]:
import os
import time
import requests
import pandas as pd
from datetime import datetime, timedelta

def fetch_pendle_apys(
    markets_dict: dict,
    output_dir: str,
    time_frame: str = 'hour',
    max_retries: int = 3
):
    """
    Fetch historical APY data for Pendle markets and save to CSV files.
    Paginates backward in 1440-hour windows.
    """
    base_url = "https://api-v2.pendle.finance/core/v2/1/markets/{}/historical-data"
    os.makedirs(output_dir, exist_ok=True)

    for symbol, info in markets_dict.items():
        pendle_address = info.get('pendle_address')
        if not pendle_address:
            print(f"Skipping {symbol}: missing pendle_address")
            continue

        print(f"Fetching data for {symbol} ({pendle_address})")
        all_results = []
        
        # Start with no end bound (get most recent data first)
        next_end = None
        limit = 1440  # API max

        while True:
            params = {
                'limit': limit,
                'time_frame': time_frame
            }
            
            # Set time window for backward pagination
            if next_end:
                end_dt = datetime.fromisoformat(next_end.replace('Z', '+00:00'))
                start_dt = end_dt - timedelta(hours=1440)
                params['timestamp_start'] = start_dt.strftime("%Y-%m-%dT%H:%M:%SZ")
                params['timestamp_end'] = end_dt.strftime("%Y-%m-%dT%H:%M:%SZ")

            # Retry logic
            retry_count = 0
            data = None
            while retry_count < max_retries:
                try:
                    response = requests.get(
                        base_url.format(pendle_address),
                        params=params,
                        timeout=30
                    )
                    response.raise_for_status()
                    data = response.json()
                    break
                except requests.exceptions.RequestException as e:
                    retry_count += 1
                    if retry_count == max_retries:
                        print(f"Error fetching data for {symbol} after {max_retries} attempts: {e}")
                        break
                    wait_time = 2 ** (retry_count - 1)
                    print(f"Retry {retry_count}/{max_retries} for {symbol} after {wait_time}s")
                    time.sleep(wait_time)

            if data is None:
                break

            results = data.get('results', [])
            if not results:
                break

            all_results.extend(results)

            # Stop if we got fewer records than limit (reached beginning)
            if len(results) < limit:
                break

            # Set next_end to earliest timestamp in current batch
            first_ts_str = results[0]['timestamp']
            first_ts = datetime.fromisoformat(first_ts_str.replace('Z', '+00:00'))
            next_end = first_ts.strftime("%Y-%m-%dT%H:%M:%SZ")

            time.sleep(0.5)

        if not all_results:
            print(f"No data found for {symbol}")
            continue

        # Convert to DataFrame
        df = pd.DataFrame(all_results)
        df = df.rename(columns={
            'baseApy': 'base_apy',
            'impliedApy': 'implied_apy',
            'underlyingApy': 'underlying_apy',
            'tvl': 'tvl'
        })

        df['datetime'] = pd.to_datetime(df['timestamp'])
        df['symbol'] = symbol
        df['market_id'] = pendle_address

        columns = ['timestamp', 'datetime', 'base_apy', 'implied_apy',
                   'underlying_apy', 'tvl', 'symbol', 'market_id']
        df = df[columns]

        filename = os.path.join(output_dir, f"{symbol}.csv")
        df.to_csv(filename, index=False)
        print(f"Saved {len(df)} records to {filename}")

In [ ]:

import json

# Read the entire file
with open('/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/common/assets_meta.json', 'r') as f:
    assets_meta = json.load(f)
market_dict = {
    "PT-siUSD-26MAR2026": {"pendle_address": "0x564f279b0226f60a40f1e4b8c596feb87c383bfa", "asset_address": "0xaF76B3AF3477E4a2cD0B7F80c3152108c19a25e5"},
    "PT-reUSD-25JUN2026": {"pendle_address": "0xf5929a1c332ceab7918a4593a43db2b9ac20095f", "asset_address": "0x3EAA0F0f0A5d3D595ae4e4b0D27f439d01c3E7b2"},

"PT-RLP-4SEP2025":	 { "pendle_address": "0x55f06992e4c3ed17df830da37644885c0c34edda", "asset_address": "" }, 
"PT-USD0++-27MAR2025":	 { "pendle_address": "0xafdc922d0059147486cc1f0f32e3a2354b0d35cc", "asset_address": "" }, 
"PT-USD0++-31OCT2024":	 { "pendle_address": "0x00b321d89a8c36b3929f20b7955080baed706d1b", "asset_address": "" }, 

'PT-USDe-25SEP2025': {'pendle_address': '0xa3336f04f7afbf26714331e395054f33b77c9b8d',
  'asset_address': ''},
 'PT-USDe-27MAR2025': {'pendle_address': '0xa3336f04f7afbf26714331e395054f33b77c9b8d',
  'asset_address': ''},
 'PT-USDe-27NOV2025': {'pendle_address': '0xa3336f04f7afbf26714331e395054f33b77c9b8d',
  'asset_address': ''},
 'PT-USDe-31JUL2025': {'pendle_address': '0xa3336f04f7afbf26714331e395054f33b77c9b8d',
  'asset_address': ''},
 'PT-USR-29MAY2025': {'pendle_address': '0x35a18cd59a214c9e797e14b1191b700eea251f6a', 'asset_address': None},
 'PT-csUSDL-31JUL2025': {'pendle_address': '0x08bf93c8f85977c64069dd34c5da7b1c636e104f', 'asset_address': None},
 'PT-lvlUSD-29MAY2025': {'pendle_address': '0xe45d2ce15abba3c67b9ff1e7a69225c855d3da82', 'asset_address': None},
 'PT-mHYPER-20NOV2025': {'pendle_address': '0x3a4204255257698e379245ef94274ef3b2907296', 'asset_address': None},
 'PT-reUSD-18DEC2025': {'pendle_address': '0xf5929a1c332ceab7918a4593a43db2b9ac20095f',
  'asset_address': ''},
 'PT-reUSD-25JUN2026': {'pendle_address': '0xf5929a1c332ceab7918a4593a43db2b9ac20095f',
  'asset_address': ''},
 'PT-sNUSD-5MAR2026': {'pendle_address': '0x6d8c4de7071d5aee27fc3a810764e62a4a00ceb9', 'asset_address': None},
 'PT-slvlUSD-25SEP2025': {'pendle_address': '0xc88ff954d42d3e11d43b62523b3357847c29377c', 'asset_address': None},
 'PT-slvlUSD-29MAY2025': {'pendle_address': '0x1c71752a6c10d66375702aafad4b6d20393702cf', 'asset_address': None},
 'PT-stcUSD-23JUL2026': {'pendle_address': '0xac24a6f0068d9701eaea76ab0b418021017f8d59',
  'asset_address': ''},
 'PT-stcUSD-29JAN2026': {'pendle_address': '0xac24a6f0068d9701eaea76ab0b418021017f8d59',
  'asset_address': ''},
 'PT-syrupUSDC-28AUG2025': {'pendle_address': '0xb327ead785574789bf4f7ef32bcdeae9513983d1',
  'asset_address': ''},
 'PT-syrupUSDC-30OCT2025': {'pendle_address': '0xb327ead785574789bf4f7ef32bcdeae9513983d1',
  'asset_address': ''},
 'PT-wstUSR-25SEP2025': {'pendle_address': '0x09fa04aac9c6d1c6131352ee950cd67ecc6d4fb9', 'asset_address': None},
 'PT-wstUSR-27MAR2025': {'pendle_address': '0x353d0b2efb5b3a7987fb06d30ad6160522d08426', 'asset_address': None}
}

market_dict = {
#     'PT-USDe-25SEP2025': {'pendle_address': '0x6d98a2b6cdbf44939362a3e99793339ba2016af4',
#   'asset_address': ''},
#  'PT-USDe-27MAR2025': {'pendle_address': '0xb451a36c8b6b2eac77ad0737ba732818143a0e25',
#   'asset_address': ''},
#  'PT-USDe-27NOV2025': {'pendle_address': '0x4eaa571eafcd96f51728756bd7f396459bb9b869',
#   'asset_address': ''},
#  'PT-USDe-31JUL2025': {'pendle_address': '0x9df192d13d61609d1852461c4850595e1f56e714',
#   'asset_address': ''},
#  'PT-USR-29MAY2025': {'pendle_address': '0x35a18cd59a214c9e797e14b1191b700eea251f6a',
#   'asset_address': ''},
#  'PT-csUSDL-31JUL2025': {'pendle_address': '0x08bf93c8f85977c64069dd34c5da7b1c636e104f',
#   'asset_address': ''},
#  'PT-lvlUSD-29MAY2025': {'pendle_address': '0xe45d2ce15abba3c67b9ff1e7a69225c855d3da82',
#   'asset_address': ''},
#  'PT-mHYPER-20NOV2025': {'pendle_address': '0x3a4204255257698e379245ef94274ef3b2907296', 'asset_address': ''},
#  'PT-reUSD-18DEC2025': {'pendle_address': '0xa8f21c9d0afb46382ead7c33e79f00ef9666e122',
#   'asset_address': ''},
#  'PT-reUSD-25JUN2026': {'pendle_address': '0xf5929a1c332ceab7918a4593a43db2b9ac20095f',
#   'asset_address': ''},
#  'PT-sNUSD-5MAR2026': {'pendle_address': '0x4bba42da555f3d8c2b441ca6d8ef9bd1ebf3bff8',
#   'asset_address': ''},
#  'PT-slvlUSD-25SEP2025': {'pendle_address': '0xc88ff954d42d3e11d43b62523b3357847c29377c',
#   'asset_address': ''},
#  'PT-slvlUSD-29MAY2025': {'pendle_address': '0x1c71752a6c10d66375702aafad4b6d20393702cf',
#   'asset_address': ''},
#  'PT-stcUSD-23JUL2026': {'pendle_address': '0xac24a6f0068d9701eaea76ab0b418021017f8d59',
#   'asset_address': ''},
#  'PT-stcUSD-29JAN2026': {'pendle_address': '0xcc781b043933c10a04409b22aada3a3d1a7f29d4',
#   'asset_address': ''},
#  'PT-syrupUSDC-28AUG2025': {'pendle_address': '0x9a63fa80b5ddfd3cab23803fdb93ad2c18f3d5aa',
#   'asset_address': ''},
#  'PT-syrupUSDC-30OCT2025': {'pendle_address': '0x8f7eddfa1a03d872da73d9588b040b608238f863',
#   'asset_address': ''},
#  'PT-wstUSR-25SEP2025': {'pendle_address': '0x09fa04aac9c6d1c6131352ee950cd67ecc6d4fb9',
#   'asset_address': ''},
#  'PT-wstUSR-27MAR2025': {'pendle_address': '0x353d0b2efb5b3a7987fb06d30ad6160522d08426',
#   'asset_address': ''}
  }


# fetch_pendle_historical_apys(
#     market_dict,
#     assets_meta,
#     "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/pt_yields",
#     start_date="2025-01-01"
# )

fetch_pendle_apys(
    market_dict,
    "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/pt_yields",
)


Fetching data for PT-stcUSD-23JUL2026 (0xac24a6f0068d9701eaea76ab0b418021017f8d59)
Saved 1636 records to /Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/pt_yields/PT-stcUSD-23JUL2026.csv
Fetching data for PT-stcUSD-29JAN2026 (0xcc781b043933c10a04409b22aada3a3d1a7f29d4)
Saved 3899 records to /Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/pt_yields/PT-stcUSD-29JAN2026.csv
Fetching data for PT-syrupUSDC-28AUG2025 (0x9a63fa80b5ddfd3cab23803fdb93ad2c18f3d5aa)
Saved 3407 records to /Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/pt_yields/PT-syrupUSDC-28AUG2025.csv
Fetching data for PT-syrupUSDC-30OCT2025 (0x8f7eddfa1a03d872da73d9588b040b608238f863)
Saved 1906 records to /Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/pt_yields/PT-syrupUSDC-30OCT2025.csv
Fetching data for PT-wstUSR-25SEP2025 (0x09fa04aac9c6d1c6131352ee950cd67ecc6d4fb9)
Saved 4574 records to /Users/yegortrussov/Documen

In [21]:
tm = """
-rw-r--r--@  1 yegortrussov  staff    1169675 Mar 15 18:10 eth_PT-RLP-4SEP2025_usdc.csv
-rw-r--r--@  1 yegortrussov  staff     744679 Mar 15 18:03 eth_PT-USD0++-27MAR2025_usdc.csv
-rw-r--r--@  1 yegortrussov  staff     161862 Mar 15 18:09 eth_PT-USD0++-31OCT2024_usdc.csv
-rw-r--r--@  1 yegortrussov  staff    1264914 Mar 15 18:02 eth_PT-USDe-25SEP2025_dai.csv
-rw-r--r--@  1 yegortrussov  staff    5362548 Mar 15 18:03 eth_PT-USDe-25SEP2025_usdc.csv
-rw-r--r--@  1 yegortrussov  staff    2063311 Mar 15 18:04 eth_PT-USDe-25SEP2025_usdt.csv
-rw-r--r--@  1 yegortrussov  staff    1163821 Mar 15 18:03 eth_PT-USDe-27MAR2025_dai.csv
-rw-r--r--@  1 yegortrussov  staff     399475 Mar 15 18:03 eth_PT-USDe-27NOV2025_usds.csv
-rw-r--r--@  1 yegortrussov  staff     689109 Mar 15 18:09 eth_PT-USDe-31JUL2025_dai.csv
-rw-r--r--@  1 yegortrussov  staff     506516 Mar 15 18:07 eth_PT-USR-29MAY2025_usdc.csv
-rw-r--r--@  1 yegortrussov  staff    2034776 Mar 15 18:06 eth_PT-csUSDL-31JUL2025_usdc.csv
-rw-r--r--@  1 yegortrussov  staff    1484933 Mar 15 18:07 eth_PT-lvlUSD-29MAY2025_usdc.csv
-rw-r--r--@  1 yegortrussov  staff     985759 Mar 15 18:06 eth_PT-mHYPER-20NOV2025_usdc.csv
-rw-r--r--@  1 yegortrussov  staff     362607 Mar 15 18:09 eth_PT-reUSD-18DEC2025_usdc.csv
-rw-r--r--@  1 yegortrussov  staff    5423688 Mar 15 18:05 eth_PT-reUSD-25JUN2026_usdc.csv
-rw-r--r--@  1 yegortrussov  staff    1006285 Mar 15 18:10 eth_PT-sNUSD-5MAR2026_usdc.csv
-rw-r--r--@  1 yegortrussov  staff    1295367 Mar 15 18:09 eth_PT-sdeUSD-1753142406_usdc.csv
-rw-r--r--@  1 yegortrussov  staff    2293950 Mar 15 18:08 eth_PT-slvlUSD-25SEP2025_usdc.csv
-rw-r--r--@  1 yegortrussov  staff    1034867 Mar 15 18:07 eth_PT-slvlUSD-29MAY2025_usdc.csv
-rw-r--r--@  1 yegortrussov  staff    1102885 Mar 15 18:06 eth_PT-stcUSD-23JUL2026_usdc.csv
-rw-r--r--@  1 yegortrussov  staff    3356490 Mar 15 18:04 eth_PT-stcUSD-29JAN2026_usdc.csv
-rw-r--r--@  1 yegortrussov  staff     509279 Mar 15 18:03 eth_PT-syrupUSDC-28AUG2025_usdc.csv
-rw-r--r--@  1 yegortrussov  staff    1384639 Mar 15 18:05 eth_PT-syrupUSDC-30OCT2025_usdc.csv
-rw-r--r--@  1 yegortrussov  staff    1250342 Mar 15 18:09 eth_PT-wstUSR-25SEP2025_usdc.csv
-rw-r--r--@  1 yegortrussov  staff      39449 Mar 15 18:05 eth_PT-wstUSR-27MAR2025_usdc.csv
-rw-r--r--@  1 yegortrussov  staff     286132 Mar 15 18:07 eth_PT-wstUSR-27MAR2025_usr.csv
"""
st = '{'
st1 = '}'
for i in tm.split("\n")[1:-1]:
    print(f'"{i.split(" ")[-1][4:-9]}":\t {st} "pendle_address": "", "asset_address": "" {st1}, ')

"PT-RLP-4SEP2025":	 { "pendle_address": "", "asset_address": "" }, 
"PT-USD0++-27MAR2025":	 { "pendle_address": "", "asset_address": "" }, 
"PT-USD0++-31OCT2024":	 { "pendle_address": "", "asset_address": "" }, 
"PT-USDe-25SEP202":	 { "pendle_address": "", "asset_address": "" }, 
"PT-USDe-25SEP2025":	 { "pendle_address": "", "asset_address": "" }, 
"PT-USDe-25SEP2025":	 { "pendle_address": "", "asset_address": "" }, 
"PT-USDe-27MAR202":	 { "pendle_address": "", "asset_address": "" }, 
"PT-USDe-27NOV2025":	 { "pendle_address": "", "asset_address": "" }, 
"PT-USDe-31JUL202":	 { "pendle_address": "", "asset_address": "" }, 
"PT-USR-29MAY2025":	 { "pendle_address": "", "asset_address": "" }, 
"PT-csUSDL-31JUL2025":	 { "pendle_address": "", "asset_address": "" }, 
"PT-lvlUSD-29MAY2025":	 { "pendle_address": "", "asset_address": "" }, 
"PT-mHYPER-20NOV2025":	 { "pendle_address": "", "asset_address": "" }, 
"PT-reUSD-18DEC2025":	 { "pendle_address": "", "asset_address": "" }, 
"PT-reUSD-25JUN

In [55]:
import requests
import re
from datetime import datetime

def fetch_pendle_markets(chain_id=1):
    """Fetch all active markets from Pendle API."""
    url = f"https://api-v2.pendle.finance/core/v1/markets/all?chainId={chain_id}"
    response = requests.get(url)
    response.raise_for_status()
    return response.json().get('markets', [])

def parse_maturity_from_name(token_name):
    """Extract underlying asset and maturity from token name like 'PT-wstUSR-27MAR2025'"""
    # Remove 'PT-' prefix
    name = token_name.replace('PT-', '')
    
    # Split by last '-' to separate asset and date
    parts = name.rsplit('-', 1)
    if len(parts) != 2:
        return name, None
    
    asset = parts[0]
    maturity_str = parts[1]
    dd = {
        "JAN": "01",
        "FEB": "02",
        "MAR": "03",
        "APR": "04",
        "MAY": "05",
        "JUN": "06",
        "JUL": "07",
        "AUG": "08",
        "SEP": "09",
        "OCT": "10",
        "NOV": "11",
        "DEC": "12"
    }
    try:
        mat = maturity_str[-4:] +"-"+ dd[maturity_str[2:2+3]] + "-" + maturity_str[:2]
    except Exception as e:
        mat=''
    return asset, mat
    
    # Parse maturity date (e.g., '27MAR2025' -> datetime)
    try:
        # Handle 2-digit year case (e.g., '25SEP202' -> '2025')
        if len(maturity_str) == 9 and maturity_str[-1].isdigit():
            maturity_str = maturity_str[:-1] + '20' + maturity_str[-1]
        maturity = datetime.strptime(maturity_str, '%d%b%Y')
        if asset == "USR":
            print("AAAAAAAAA", maturity)
        return asset, maturity
    except:
        return asset, None

def find_market_for_token(markets, token_name):
    """
    Find market matching token name.
    Returns dict with pendle_address and asset_address.
    """
    token_asset, target_maturity = parse_maturity_from_name(token_name)
    print('\n')
    print("Maturity", target_maturity)
    if not token_asset:
        return {'pendle_address': None, 'asset_address': None}
    
    for m in markets:
        market_name = m.get('name', '')
        if not market_name:
            continue
        
        # Match by asset name
        if market_name.lower() == token_asset.lower():
            # If maturity specified, compare expiry dates
            if target_maturity and m.get('expiry'):
                expiry_str = m['expiry'].split('T')[0]  # '2025-03-27'
                # expiry_date = datetime.strptime(expiry_str, '%Y-%m-%d').date()
                # target_date = target_maturity.date()
                print(expiry_str, "--", target_maturity, market_name, token_asset)
                if expiry_str != target_maturity:
                    continue
            
            # Extract asset address from underlyingAsset (format: chainId-address)
            underlying = m.get('underlyingAsset', '')
            asset_address = underlying.split('-')[-1] if underlying and '-' in underlying else None
            
            return {
                'pendle_address': m['address'],
                'asset_address': ""
            }
    
    return {'pendle_address': None, 'asset_address': None}

def fill_market_addresses(token_list):
    """Fill pendle_address and asset_address for all tokens."""
    print("Fetching markets from API...")
    markets = fetch_pendle_markets()
    print(f"Fetched {len(markets)} markets")
    
    for token_name in token_list:
        result = find_market_for_token(markets, token_name)
        token_list[token_name]['pendle_address'] = result['pendle_address']
        token_list[token_name]['asset_address'] = result['asset_address']
        status = "✓" if result['pendle_address'] else "✗"
        print(f"{status} {token_name}: pendle={result['pendle_address']}, asset={result['asset_address']}")
    
    return token_list

# Your token list
tokens = {
    "PT-USDe-25SEP2025": {"pendle_address": "", "asset_address": ""},
    "PT-USDe-27MAR2025": {"pendle_address": "", "asset_address": ""},
    "PT-USDe-27NOV2025": {"pendle_address": "", "asset_address": ""},
    "PT-USDe-31JUL2025": {"pendle_address": "", "asset_address": ""},
    "PT-USR-29MAY2025": {"pendle_address": "", "asset_address": ""},
    "PT-csUSDL-31JUL2025": {"pendle_address": "", "asset_address": ""},
    "PT-lvlUSD-29MAY2025": {"pendle_address": "", "asset_address": ""},
    "PT-mHYPER-20NOV2025": {"pendle_address": "", "asset_address": ""},
    "PT-reUSD-18DEC2025": {"pendle_address": "", "asset_address": ""},
    "PT-reUSD-25JUN2026": {"pendle_address": "", "asset_address": ""},
    "PT-sNUSD-5MAR2026": {"pendle_address": "", "asset_address": ""},
    "PT-sdeUSD-1753142406": {"pendle_address": "", "asset_address": ""},
    "PT-slvlUSD-25SEP2025": {"pendle_address": "", "asset_address": ""},
    "PT-slvlUSD-29MAY2025": {"pendle_address": "", "asset_address": ""},
    "PT-stcUSD-23JUL2026": {"pendle_address": "", "asset_address": ""},
    "PT-stcUSD-29JAN2026": {"pendle_address": "", "asset_address": ""},
    "PT-syrupUSDC-28AUG2025": {"pendle_address": "", "asset_address": ""},
    "PT-syrupUSDC-30OCT2025": {"pendle_address": "", "asset_address": ""},
    "PT-wstUSR-25SEP2025": {"pendle_address": "", "asset_address": ""},
    "PT-wstUSR-27MAR2025": {"pendle_address": "", "asset_address": ""},
}

# Run the filling
filled_tokens = fill_market_addresses(tokens)

filled_tokens

Fetching markets from API...
Fetched 419 markets


Maturity 2025-09-25
2024-07-25 -- 2025-09-25 USDe USDe
2024-10-24 -- 2025-09-25 USDe USDe
2025-11-27 -- 2025-09-25 USDe USDe
2025-09-25 -- 2025-09-25 USDe USDe
✓ PT-USDe-25SEP2025: pendle=0x6d98a2b6cdbf44939362a3e99793339ba2016af4, asset=


Maturity 2025-03-27
2024-07-25 -- 2025-03-27 USDe USDe
2024-10-24 -- 2025-03-27 USDe USDe
2025-11-27 -- 2025-03-27 USDe USDe
2025-09-25 -- 2025-03-27 USDe USDe
2024-12-26 -- 2025-03-27 USDe USDe
2024-06-27 -- 2025-03-27 USDe USDe
2025-07-31 -- 2025-03-27 USDe USDe
2026-05-07 -- 2025-03-27 USDe USDe
2026-02-05 -- 2025-03-27 USDe USDe
2024-04-04 -- 2025-03-27 USDe USDe
2025-03-27 -- 2025-03-27 USDe USDe
✓ PT-USDe-27MAR2025: pendle=0xb451a36c8b6b2eac77ad0737ba732818143a0e25, asset=


Maturity 2025-11-27
2024-07-25 -- 2025-11-27 USDe USDe
2024-10-24 -- 2025-11-27 USDe USDe
2025-11-27 -- 2025-11-27 USDe USDe
✓ PT-USDe-27NOV2025: pendle=0x4eaa571eafcd96f51728756bd7f396459bb9b869, asset=


Maturity 2025-07

{'PT-USDe-25SEP2025': {'pendle_address': '0x6d98a2b6cdbf44939362a3e99793339ba2016af4',
  'asset_address': ''},
 'PT-USDe-27MAR2025': {'pendle_address': '0xb451a36c8b6b2eac77ad0737ba732818143a0e25',
  'asset_address': ''},
 'PT-USDe-27NOV2025': {'pendle_address': '0x4eaa571eafcd96f51728756bd7f396459bb9b869',
  'asset_address': ''},
 'PT-USDe-31JUL2025': {'pendle_address': '0x9df192d13d61609d1852461c4850595e1f56e714',
  'asset_address': ''},
 'PT-USR-29MAY2025': {'pendle_address': '0x35a18cd59a214c9e797e14b1191b700eea251f6a',
  'asset_address': ''},
 'PT-csUSDL-31JUL2025': {'pendle_address': '0x08bf93c8f85977c64069dd34c5da7b1c636e104f',
  'asset_address': ''},
 'PT-lvlUSD-29MAY2025': {'pendle_address': '0xe45d2ce15abba3c67b9ff1e7a69225c855d3da82',
  'asset_address': ''},
 'PT-mHYPER-20NOV2025': {'pendle_address': None, 'asset_address': None},
 'PT-reUSD-18DEC2025': {'pendle_address': '0xa8f21c9d0afb46382ead7c33e79f00ef9666e122',
  'asset_address': ''},
 'PT-reUSD-25JUN2026': {'pendle_add

In [39]:
for i in filled_tokens:
    print(i)

PT-USDe-25SEP2025
PT-USDe-27MAR2025
PT-USDe-27NOV2025
PT-USDe-31JUL2025
PT-USR-29MAY2025
PT-csUSDL-31JUL2025
PT-lvlUSD-29MAY2025
PT-mHYPER-20NOV2025
PT-reUSD-18DEC2025
PT-reUSD-25JUN2026
PT-sNUSD-5MAR2026
PT-sdeUSD-1753142406
PT-slvlUSD-25SEP2025
PT-slvlUSD-29MAY2025
PT-stcUSD-23JUL2026
PT-stcUSD-29JAN2026
PT-syrupUSDC-28AUG2025
PT-syrupUSDC-30OCT2025
PT-wstUSR-25SEP2025
PT-wstUSR-27MAR2025
